# build Vector DB: Pinecone

**Architecture:**
- Vector DB: Pinecone (1 index, 3 namespaces: `rent` / `labor` ,`NDA` )
- Embeddings: OpenAI `text-embedding-3-large` (3072 dim)
- LLM: OpenAI `gpt-4o` (large)
- Data: JSON files


##  Install dependencies

In [1]:
!pip install -q openai pinecone tiktoken tenacity
!pip install pinecone --break-system-packages -q
!pip uninstall pinecone pinecone-client -y
!pip install pinecone==3.2.2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 33.1 MB/s eta 0:00:00
Found existing installation: pinecone 8.1.2
Uninstalling pinecone-8.1.2:
  Successfully uninstalled pinecone-8.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.9/215.9 kB 11.4 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY']   = '.....'
os.environ['PINECONE_API_KEY'] = '.....'

INDEX_NAME  = 'ragindex'
DIMENSION   = 3072
METRIC      = 'cosine'

NS_RENT  = 'rent'
NS_LABOR = 'labor'
NS_NDA = 'NDA'

print(' Keys loaded')

 Keys loaded


## Upload & load your JSON files

In [13]:
import json
from google.colab import files

uploaded = files.upload()

rent_raw = []
labor_raw = []
NDA_raw = []

for fname in uploaded:
    content = uploaded[fname].decode('utf-8')
    data = json.loads(content)

    if 'rent' in fname.lower():
        rent_raw = data
        print(f' Rent: {len(rent_raw)} records')

    elif 'labor' in fname.lower():
        labor_raw = data
        print(f' Labor: {len(labor_raw)} records')

    else:
        NDA_raw = data
        print(f' NDA: {len(NDA_raw)} records')

Saving nda_clauses_master_clean1.json to nda_clauses_master_clean1.json
Saving labor_contracts_final_clean.json to labor_contracts_final_clean.json
Saving rental_contracts_master_clean.json to rental_contracts_master_clean.json
 NDA: 65 records
 Labor: 266 records
 Rent: 76 records


In [15]:
print(rent_raw[0])
print(labor_raw[0])
print(NDA_raw[0])

{'field': 'contract_duration', 'title_ar': 'مدة العقد', 'category': 'contract_terms', 'contract_subtype': 'both', 'required': 'نعم', 'risk_if_missing': 'غموض في تحديد انتهاء العقد والتجديد', 'description': 'تحديد مدة الإيجار بدقة (شهور أو سنوات) وتاريخ البداية والنهاية', 'keywords': 'مدة العقد، سنة، شهر، انتهاء، مدة الإيجار، duration، lease term، contract period، rental period', 'reference': 'نموذج العقد الإيجاري الموحد (منصة إيجار) — البند 1 (بيانات العقد)'}
{'field': 'allowances', 'title_ar': 'البدلات', 'category': 'compensation', 'required': 'لا', 'risk_if_missing': 'حرمان من مستحقات إضافية', 'description': 'تحديد بدل سكن أو نقل أو غيره', 'keywords': 'بدل، allowances', 'reference': 'المادة 18'}
{'field': 'confidential_information_definition', 'title_ar': 'تعريف المعلومات السرية', 'category': 'confidentiality_restrictions', 'required': 'نعم', 'risk_if_missing': 'نزاع حول ما يُعدّ سرياً وما لا يُعدّ', 'description': 'تحديد ما تعتبره الاتفاقية معلومات سرية كالبيانات المالية والتقنية وق

##  Clean & validate

In [25]:
def clean_records(raw):
    cleaned = []
    skipped = 0

    print(f' Total records: {len(raw)}')

    for i, rec in enumerate(raw):
        text_parts = [
            rec.get('title_ar', ''),
            rec.get('description', ''),
            rec.get('keywords', '')
        ]

        content = ' | '.join([t for t in text_parts if t]).strip()

        if not content:
            skipped += 1
            continue

        cleaned.append({
            'id': rec.get('field', f'auto_{i}'),
            'text': content,
            'metadata': {
                'category': rec.get('category'),
                'required': rec.get('required'),
                'risk': rec.get('risk_if_missing'),
                'reference': rec.get('reference'),
                'contract_subtype': rec.get('contract_subtype')
            }
        })


    if cleaned:
        print(' Sample:')
        print(cleaned[0])

    return cleaned

print(f'rent → {len(rent_docs)} | labor → {len(labor_docs)} | NDA → {len(nda_docs)}')

rent → 76 | labor → 266 | NDA → 65


## Chunking

In [22]:
import tiktoken
import json

ENC          = tiktoken.get_encoding('cl100k_base')
CHUNK_TOKENS = 512
OVERLAP      = 50

def chunk_document(doc: dict, chunk_tokens: int = CHUNK_TOKENS, overlap: int = OVERLAP) -> list:
    """
    Split a document into token-bounded chunks with sliding overlap.
    Preserves all metadata on every chunk.
    Returns list of chunk dicts ready for embedding.
    """
    text   = doc['text']
    tokens = ENC.encode(text)

    if len(tokens) <= chunk_tokens:
        return [{'id': f"{doc['id']}_c0", 'text': text, 'metadata': doc['metadata']}]

    chunks = []
    start  = 0
    idx    = 0
    while start < len(tokens):
        end        = min(start + chunk_tokens, len(tokens))
        chunk_text = ENC.decode(tokens[start:end])
        chunks.append({
            'id'      : f"{doc['id']}_c{idx}",
            'text'    : chunk_text,
            'metadata': {**doc['metadata'], 'chunk_index': idx}
        })
        start += chunk_tokens - overlap
        idx   += 1

    return chunks


def chunk_all(docs: list) -> list:
    result = []
    for doc in docs:
        result.extend(chunk_document(doc))
    return result


rent_chunks  = chunk_all(rent_docs)
labor_chunks = chunk_all(labor_docs)
nda_chunks   = chunk_all(nda_docs)

print(f' Chunks created:')
print(f'   rent  → {len(rent_chunks)} chunks')
print(f'   labor → {len(labor_chunks)} chunks')
print(f'   NDA   → {len(nda_chunks)} chunks')

print('Sample rent chunk:')
print(json.dumps(rent_chunks[0], ensure_ascii=False, indent=2))

print('Sample NDA chunk:')
print(json.dumps(nda_chunks[0], ensure_ascii=False, indent=2))

 Chunks created:
   rent  → 76 chunks
   labor → 266 chunks
   NDA   → 65 chunks
Sample rent chunk:
{
  "id": "contract_duration_c0",
  "text": "مدة العقد | تحديد مدة الإيجار بدقة (شهور أو سنوات) وتاريخ البداية والنهاية | مدة العقد، سنة، شهر، انتهاء، مدة الإيجار، duration، lease term، contract period، rental period",
  "metadata": {
    "category": "contract_terms",
    "required": "نعم",
    "risk": "غموض في تحديد انتهاء العقد والتجديد",
    "reference": "نموذج العقد الإيجاري الموحد (منصة إيجار) — البند 1 (بيانات العقد)",
    "contract_subtype": "both"
  }
}
Sample NDA chunk:
{
  "id": "confidential_information_definition_c0",
  "text": "تعريف المعلومات السرية | تحديد ما تعتبره الاتفاقية معلومات سرية كالبيانات المالية والتقنية وقوائم العملاء والمعلومات التشغيلية. | معلومات سرية، تعريف، بيانات، قوائم العملاء، معلومات تقنية، confidential information، definition، trade secret، proprietary information",
  "metadata": {
    "category": "confidentiality_restrictions",
    "required": "نعم",

##  Embed with OpenAI text-embedding-3-large

In [24]:
import time
import openai
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])

EMBED_MODEL  = 'text-embedding-3-large'
BATCH_SIZE   = 100

@retry(
    retry=retry_if_exception_type((openai.RateLimitError, openai.APIError)),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    stop=stop_after_attempt(5)
)
def embed_batch(texts: list) -> list:
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )
    return [item.embedding for item in response.data]


def embed_chunks(chunks: list, label: str = '') -> list:

    vectors = []
    total   = len(chunks)

    for i in range(0, total, BATCH_SIZE):
        batch  = chunks[i : i + BATCH_SIZE]
        texts  = [c['text'] for c in batch]
        embeds = embed_batch(texts)

        for chunk, emb in zip(batch, embeds):
            vectors.append({
                'id'      : chunk['id'],
                'values'  : emb,
                'metadata': {**chunk['metadata'], 'text': chunk['text']}
            })

        pct = min(i + BATCH_SIZE, total)
        print(f'  [{label}] Embedded {pct}/{total}...')
        time.sleep(0.3)

    return vectors


print(' Embedding rent chunks...')
rent_vectors = embed_chunks(rent_chunks, 'rent')

print(' Embedding labor chunks...')
labor_vectors = embed_chunks(labor_chunks, 'labor')

print(' Embedding NDA chunks...')
nda_vectors = embed_chunks(nda_chunks, 'NDA')

print(f' Total vectors ready: {len(rent_vectors) + len(labor_vectors) + len(nda_vectors)}')

 Embedding rent chunks...
  [rent] Embedded 76/76...
 Embedding labor chunks...
  [labor] Embedded 100/266...
  [labor] Embedded 200/266...
  [labor] Embedded 266/266...
 Embedding NDA chunks...
  [NDA] Embedded 65/65...
 Total vectors ready: 407


##  Create Pinecone index

In [26]:
from pinecone import Pinecone, ServerlessSpec
import time

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

INDEX_NAME   = 'ragindex'
DIMENSION   = 3072
METRIC      = 'cosine'

existing = [i.name for i in pc.list_indexes()]

if INDEX_NAME not in existing:
    print(f' Creating index: {INDEX_NAME}...')
    pc.create_index(
        name      = INDEX_NAME,
        dimension = DIMENSION,
        metric    = METRIC,
        spec      = ServerlessSpec(cloud='aws', region='us-east-1')
    )
    while not pc.describe_index(INDEX_NAME).status['ready']:
        time.sleep(3)
    print('Index ready!')
else:
    print(f' Index "{INDEX_NAME}" already exists')

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

 Index "ragindex" already exists
{'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}


In [29]:
def clean_metadata(metadata: dict):
    cleaned = {}

    for k, v in metadata.items():
        if v is None:
            continue

        if isinstance(v, list):
            cleaned[k] = [str(i) for i in v if i is not None]
        else:
            cleaned[k] = v

    return cleaned

## Upsert vectors into their namespaces

In [31]:
UPSERT_BATCH = 100
NS_RENT  = "rent"
NS_LABOR = "labor"
NS_NDA   = "nda"


def upsert_to_namespace(vectors: list, namespace: str):
    total = len(vectors)

    for i in range(0, total, UPSERT_BATCH):
        batch = vectors[i : i + UPSERT_BATCH]

        index.upsert(
            vectors=[
                (v['id'], v['values'], clean_metadata(v['metadata']))
                for v in batch
            ],
            namespace=namespace
        )

        pct = min(i + UPSERT_BATCH, total)
        print(f'  [{namespace}] Upserted {pct}/{total}...')

        time.sleep(0.2)


print(f' Upserting to namespace: "{NS_RENT}"...')
upsert_to_namespace(rent_vectors, NS_RENT)

print(f' Upserting to namespace: "{NS_LABOR}"...')
upsert_to_namespace(labor_vectors, NS_LABOR)

print(f' Upserting to namespace: "{NS_NDA}"...')
upsert_to_namespace(nda_vectors, NS_NDA)

time.sleep(5)

print(' Final index stats:')
print(index.describe_index_stats())

 Upserting to namespace: "rent"...
  [rent] Upserted 76/76...
 Upserting to namespace: "labor"...
  [labor] Upserted 100/266...
  [labor] Upserted 200/266...
  [labor] Upserted 266/266...
 Upserting to namespace: "nda"...
  [nda] Upserted 65/65...
 Final index stats:
{'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {'labor': {'vector_count': 266},
                'nda': {'vector_count': 65},
                'rent': {'vector_count': 76}},
 'total_vector_count': 407}


##  Query

In [40]:
SCORE_THRESHOLD = 0.50
TOP_K           = 5

SYSTEM_PROMPT = """\
أنت مساعد قانوني متخصص في قوانين المملكة العربية السعودية.
أجب فقط بناءً على المعلومات الواردة في السياق المقدم.
إذا لم تجد إجابة كافية في السياق، قل بوضوح: "لا توجد معلومات كافية في قاعدة البيانات للإجابة على هذا السؤال."
لا تخترع أو تستنتج معلومات غير موجودة في السياق.
اذكر المصدر أو رقم البند عند الإجابة إذا كان متوفراً.
كن دقيقاً وموجزاً.
"""


def detect_namespace(query: str) -> str:
    """
    Simple keyword-based routing to the correct namespace.
    Returns 'rent', 'labor', or 'both'.
    For production: replace with an LLM classifier call.
    """

    rent_keywords  = [
        'إيجار', 'مستأجر', 'مؤجر', 'إخلاء', 'عقار', 'شقة',
        'تجديد عقد', 'rent', 'tenant', 'landlord'
    ]

    labor_keywords = [
        'عمل', 'عامل', 'صاحب عمل', 'راتب', 'إنهاء خدمة',
        'مكافأة', 'labor', 'employee', 'salary', 'termination'
    ]

    nda_keywords = [
        'سرية', 'عدم الإفصاح', 'معلومات سرية', 'confidential',
        'NDA', 'non disclosure', 'اتفاقية سرية', 'عدم الكشف',
        'بيانات سرية', 'معلومات حساسة'
    ]

    q_lower     = query.lower()
    rent_match  = any(kw in q_lower for kw in rent_keywords)
    labor_match = any(kw in q_lower for kw in labor_keywords)
    nda_match   = any(kw in q_lower for kw in nda_keywords)

    if nda_match:
        return NS_NDA
    elif rent_match and not labor_match:
        return NS_RENT
    elif labor_match and not rent_match:
        return NS_LABOR
    else:
        return NS_NDA


def retrieve_chunks(query: str, namespace: str, top_k: int = TOP_K) -> list:
    """Embed query and retrieve top-k chunks from Pinecone namespace."""

    q_vector   = embed_batch([query])[0]
    namespaces = [namespace]

    all_chunks = []

    for ns in namespaces:
        result = index.query(
            vector          = q_vector,
            top_k           = top_k,
            namespace       = ns,
            include_metadata= True
        )

        for match in result.matches:
            if match.score >= SCORE_THRESHOLD:
                all_chunks.append({
                    'id'       : match.id,
                    'score'    : round(match.score, 3),
                    'text'     : match.metadata.get('text', ''),
                    'namespace': ns,
                    'category' : match.metadata.get('category', ''),
                    'reference': match.metadata.get('reference', ''),
                })

    all_chunks.sort(key=lambda x: x['score'], reverse=True)
    return all_chunks


def build_context(chunks: list) -> str:
    """Format retrieved chunks into a numbered context block."""

    if not chunks:
        return 'لا توجد نتائج ذات صلة.'

    parts = []
    for i, c in enumerate(chunks, 1):
        ref = f" [{c['reference']}]" if c.get('reference') else ''
        parts.append(
            f"[{i}] (score={c['score']}, ns={c['namespace']}{ref})\n{c['text']}"
        )

    return '\n\n'.join(parts)


def rag_query(question: str, verbose: bool = False) -> str:
    """
    Full RAG pipeline:
    1. Detect namespace
    2. Retrieve relevant chunks
    3. Generate grounded answer with GPT-4o
    """

    namespace = detect_namespace(question)

    if verbose:
        print(f' Routing to namespace: {namespace}')

    chunks = retrieve_chunks(question, namespace)

    if verbose:
        print(f' Retrieved {len(chunks)} chunks above threshold {SCORE_THRESHOLD}')
        for c in chunks:
            print(f' score={c["score"]} | ns={c["namespace"]} | {c["text"][:80]}...')

    if not chunks:
        return 'لا توجد معلومات كافية في قاعدة البيانات للإجابة على هذا السؤال.'

    context  = build_context(chunks)

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f'السياق:\n{context}\n\nالسؤال: {question}'}
    ]

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=messages,
        temperature=0.1,
        max_tokens=1000
    )

    return response.choices[0].message.content


print(' RAG  ready ')

 RAG  ready 


## Test

In [33]:
q1 = 'ما هي مدة الإشعار المطلوبة لعدم تجديد عقد الإيجار؟'
print(f' السؤال: {q1}')
print('─' * 60)
ans1 = rag_query(q1, verbose=True)
print('\ الإجابة:')
print(ans1)

<>:5: SyntaxWarning: invalid escape sequence '\ '
<>:5: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_4803/1873179768.py:5: SyntaxWarning: invalid escape sequence '\ '
  print('\ الإجابة:')


 السؤال: ما هي مدة الإشعار المطلوبة لعدم تجديد عقد الإيجار؟
────────────────────────────────────────────────────────────
 Routing to namespace: {namespace}
 Retrieved 5 chunks above threshold 0.5
 score=0.707 | ns=rent | مهلة إشعار عدم التجديد | يجب إشعار الطرف الآخر بعدم الرغبة في التجديد قبل 60 يوم...
 score=0.607 | ns=rent | التجديد التلقائي | يتجدد عقد الإيجار تلقائياً ما لم يُشعر أحد الطرفين الآخر بعدم...
 score=0.593 | ns=rent | الإخلاء بعدم السداد — مهلة 30 يوماً | إذا تأخر المستأجر في دفع الإيجار يحق للمؤج...
 score=0.581 | ns=rent | مهلة الإشعار للاستخدام الشخصي (365 يومًا) | إذا أراد المؤجر إخلاء العقار لاستخدا...
 score=0.556 | ns=rent | شرط الإخلاء الفوري دون إشعار | أي بند يتيح للمؤجر إخلاء المستأجر فوراً دون إشعار...
\ الإجابة:
مدة الإشعار المطلوبة لعدم تجديد عقد الإيجار هي 60 يوماً قبل انتهاء العقد كحد أدنى. [المصدر: نموذج العقد الإيجاري الموحد (منصة إيجار) — المادة الثالثة، البند 3/3]


In [34]:
q2 = 'ما هي حقوق العامل عند إنهاء الخدمة؟'
print(f' السؤال: {q2}')
print('─' * 60)
ans2 = rag_query(q2, verbose=True)
print('\ الإجابة:')
print(ans2)

<>:5: SyntaxWarning: invalid escape sequence '\ '
<>:5: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_4803/3533646114.py:5: SyntaxWarning: invalid escape sequence '\ '
  print('\ الإجابة:')


 السؤال: ما هي حقوق العامل عند إنهاء الخدمة؟
────────────────────────────────────────────────────────────
 Routing to namespace: {namespace}
 Retrieved 5 chunks above threshold 0.5
 score=0.589 | ns=labor | مكافأة نهاية الخدمة | يستحق العامل مكافأة نهاية خدمة تحسب وفق مدة خدمته في العمل...
 score=0.531 | ns=labor | تعويض الإنهاء دون سبب مشروع | تعويض إضافي يعادل أجر 90 يومًا في حال الإنهاء دون ...
 score=0.525 | ns=labor | التعويض عن رصيد الإجازة | استحقاق مقابل مالي عن الإجازات غير المستخدمة عند انتها...
 score=0.514 | ns=labor | حقوق العمالة المقدمة للغير | حماية حقوق العمالة في عقود تقديم الخدمات | عمالة، ط...
 score=0.506 | ns=labor | حق العامل في ترك العمل فورًا | يجوز للعامل ترك العمل دون إشعار مع احتفاظه بحقوقه...
\ الإجابة:
عند إنهاء الخدمة، يحق للعامل في المملكة العربية السعودية الحصول على عدة حقوق، منها:

1. **مكافأة نهاية الخدمة**: يستحق العامل مكافأة نهاية خدمة تُحسب وفق مدة خدمته في العمل. [المادة 84]

2. **تعويض الإنهاء دون سبب مشروع**: إذا تم إنهاء خدمة العامل دون سبب مش